In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import sys; sys.path.append("../../../"); sys.path.append("../../utils")
import csv
import json
import pandas as pd

from pathlib import Path

from kb_graph import KBGraph
from kb_mapper import KBMapper
from mapping_utils import pending, run_chunked

In [ ]:
preprocessed_jobs = pd.read_csv("mapping/translated.csv")
preprocessed_jobs.head(10)

In [ ]:
esco_g = KBGraph.load(Path("../../../cache/occSkill_graph.pkl"))

job_categories_uri = [uri for uri, _ in esco_g.nodes(properties={"type": "occupation", "level": 3})]
job_categories = [esco_g.get_label(i) for i in job_categories_uri]

assert all(esco_g.get_level(i) == 3 for i in job_categories_uri)

In [ ]:
MAPPER = KBMapper(labels=job_categories, node="occupation", processed_dir=Path("../../../sources/ESCO/processed"))

In [ ]:
def extract_title(row):
    """Map the translated title to an ESCO occupation; return the aggregated row and the audit row."""
    mapped = MAPPER.map_skill(row["eng_title"], context=row["eng_description"])
    return (
        {"id_j": row["id_j"], "raw_occupation": row["eng_title"], "esco_occupation": mapped["kb_label"]},
        {"id_j": row["id_j"], "raw_skill": row["eng_title"], **mapped},
    )

In [ ]:
AGG = ("mapping/occupation_map.csv", ["id_j", "raw_occupation", "esco_occupation"])
DET = ("mapping/occupation_map_detailed.csv",
       ["id_j", "raw_skill", "kb_label", "semantic_score", "semantic_rank",
        "candidate_list", "decision_source", "confidence", "mapping_reason"])

todo = pending(preprocessed_jobs, AGG, DET)

In [ ]:
run_chunked(todo, extract_title, AGG, DET)

In [ ]:

agg = pd.read_csv(AGG[0])
det = pd.read_csv(DET[0])

def _top1(s):
    if pd.isna(s): return None
    cands = json.loads(s)
    return cands[0] if cands else None

top1_by_id = det.set_index("id_j")["candidate_list"].map(_top1)

mask = agg["esco_occupation"].isna()
agg.loc[mask, "esco_occupation"] = agg.loc[mask, "id_j"].map(top1_by_id)

print(f"NO_MATCH filled with the top-1 embedding: {mask.sum()} | esco_occupation still NaN: {agg['esco_occupation'].isna().sum()}")
agg.to_csv(AGG[0], quoting=csv.QUOTE_ALL, index=False)